# Product data analysis using Plotly

In [10]:

import pandas as pd
import plotly.express as px
from IPython.display import display, HTML
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv



In [11]:

load_dotenv()
db_name = os.getenv('DB_NAME')
db_user = os.getenv('DB_USER')
db_password = os.getenv('DB_PASSWORD')
db_host = os.getenv('DB_HOST')
db_port = os.getenv('DB_PORT')

engine = create_engine(f"postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}")

## Read products data from database


In [12]:

query = 'SELECT * FROM products'
df_products = pd.read_sql_query(query, engine)

print("Products:")
display(df_products.head())

Products:


,product_id,product_name,category,price,cost,stock_quantity,supplier
0,PRD_001,Smart Webcam,Electronics,125.21,91.76,322,TD SYNNEX
1,PRD_002,Smart Speaker,Electronics,336.32,191.26,469,TD SYNNEX
2,PRD_003,Rechargeable Speaker,Electronics,150.81,79.02,574,Tech Data
3,PRD_004,Wireless Router,Electronics,149.79,93.68,597,Exertis
4,PRD_005,4K Monitor,Electronics,196.94,147.55,141,Tech Data


### Basic product information displayed as HTML

In [13]:

number_of_products = len(df_products)
number_of_categories = df_products['category'].nunique()
cheapest_product = df_products.loc[df_products['price'].idxmin()]
most_expensive_product = df_products.loc[df_products['price'].idxmax()]
mean_product_price = df_products['price'].mean()

product_info_html = f'''<div style="background:#222; color:white; padding:20px; border-radius:10px; text-align:left; font-size:1.2em; margin-top:20px;">
<strong>Product Summary</strong><br>
<ul style="list-style:none; padding-left:0;">
<li><span style="color:#ffd700;">Number of products:</span> {number_of_products}</li>
<li><span style="color:#ffd700;">Number of categories:</span> {number_of_categories}</li>
<li><span style="color:#ffd700;">Cheapest product:</span> {cheapest_product['product_name']} (£{cheapest_product['price']})</li>
<li><span style="color:#ffd700;">Most expensive product:</span> {most_expensive_product['product_name']} (£{most_expensive_product['price']})</li>
<li><span style="color:#ffd700;">Average product price:</span> £{mean_product_price:.2f}</li>
</ul>
</div>'''
display(HTML(product_info_html))

### Bar chart of product count per category

In [14]:

category_counts = df_products['category'].value_counts().sort_index()
fig_bar = px.bar(
    x=category_counts.index,
    y=category_counts.values,
    labels={'x': 'Category', 'y': 'Number of Products'},
    title='Product Count by Category',
    text=category_counts.values,
    color=category_counts.index,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_bar.update_traces(texttemplate='%{text}', textposition='inside', textfont_color='white')
fig_bar.update_layout(
    xaxis_title='',
    yaxis_title='',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
)
fig_bar.show()


### Pie chart of category distribution


In [15]:

category_counts = df_products['category'].value_counts().sort_index()
fig_pie = px.pie(
    values=category_counts.values,
    names=category_counts.index,
    title='Product Category Distribution',
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig_pie.update_traces(textinfo='percent+label', textfont_color='white')
fig_pie.update_layout(
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig_pie.show()

### Chart of stock held per category

In [16]:

category_stock = df_products.groupby('category')['stock_quantity'].sum().sort_values(ascending=False)
category_stock_value = df_products.groupby('category').apply(lambda x: (x['stock_quantity'] * x['price']).sum()).sort_values(ascending=False)
overall_stock_value = (df_products['stock_quantity'] * df_products['price']).sum()
fig_stock = px.bar(
    x=category_stock.index,
    y=category_stock.values,
    labels={'x': '', 'y': ''},
    title='Total Stock Held per Category',
    text=category_stock.values,
    color=category_stock.index,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig_stock.update_traces(texttemplate='%{text}', textposition='inside', textfont_color='white')
fig_stock.update_layout(
    xaxis_title='',
    yaxis_title='',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig_stock.show()


## Bar chart of category stock values

In [17]:


fig_stock_value = px.bar(
    x=category_stock_value.index,
    y=category_stock_value.values,
    labels={'x': 'Category', 'y': 'Stock Value (£)'},
    title='Stock Value per Category',
    text=category_stock_value.values,
    color=category_stock_value.index,
    color_discrete_sequence=px.colors.qualitative.Set2
    )
fig_stock_value.update_traces(texttemplate='£%{text:,.2f}', textposition='inside', textfont_color='white')
fig_stock_value.update_layout(
    xaxis_title='Category',
    yaxis_title='Stock Value (£)',
    plot_bgcolor='#222',
    paper_bgcolor='#222',
    font_color='white',
    title_font_color='white',
    legend_font_color='white',
    xaxis=dict(tickfont=dict(color='white')),
    yaxis=dict(tickfont=dict(color='white'))
    )
fig_stock_value.show()


### Total value of company stock

In [18]:

overall_stock_value_html = f'''<div style="background:#222; color:white; padding:20px; border-radius:10px; text-align:center; font-size:2em; margin-top:20px;">
<strong>Overall Stock Value</strong><br>
<span style="font-size:2.5em; color:#ffd700;">£{overall_stock_value:,.2f}</span>
</div>'''
display(HTML(overall_stock_value_html))